**Import libraries**

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import notebookutils
import uuid
import re
from datetime import datetime


StatementMeta(, a16e5de9-71f8-4f68-9b09-50316f54cb30, 4, Finished, Available, Finished, False)

**Define constants**

In [3]:
INCOMING_DIR = "Files/Incoming"
BRONZE_ARCHIVE_BASE = "Files/BronzeArchive"

BRONZE_CUSTOMER_TABLE = "dbo.customer_master_bronze_raw"
BRONZE_SALES_TABLE    = "dbo.sales_transactions_bronze_raw"

SILVER_CUSTOMER_TABLE = "dbo.customer_master_silver"
SILVER_SALES_TABLE    = "dbo.sales_transactions_silver"

NOTEBOOK_NAME = "Customer_To_Sales_Silver_Notebook"
RUN_ID = str(uuid.uuid4())

FILES_TO_PROCESS = []

# Optional reset flags
RESET_BRONZE = False
RESET_SILVER = False



StatementMeta(, a16e5de9-71f8-4f68-9b09-50316f54cb30, 5, Finished, Available, Finished, False)

**Create Bronze and Silver tables if they do not exist**

In [20]:
DeltaTable.createIfNotExists(spark) \
    .tableName(BRONZE_CUSTOMER_TABLE) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("FullName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("City", StringType()) \
    .addColumn("MembershipTier", StringType()) \
    .addColumn("RecordDate", StringType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("SourceFilePath", StringType()) \
    .addColumn("ArchivePath", StringType()) \
    .addColumn("IngestedAt", TimestampType()) \
    .addColumn("RunId", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(BRONZE_SALES_TABLE) \
    .addColumn("OrderID", StringType()) \
    .addColumn("OrderLine", IntegerType()) \
    .addColumn("OrderDate", StringType()) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("ProductID", StringType()) \
    .addColumn("ProductName", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", DoubleType()) \
    .addColumn("Tax", DoubleType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("SourceFilePath", StringType()) \
    .addColumn("ArchivePath", StringType()) \
    .addColumn("IngestedAt", TimestampType()) \
    .addColumn("RunId", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(SILVER_CUSTOMER_TABLE) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("FullName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("City", StringType()) \
    .addColumn("MembershipTier", StringType()) \
    .addColumn("RecordDate", DateType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(SILVER_SALES_TABLE) \
    .addColumn("OrderID", StringType()) \
    .addColumn("OrderLine", IntegerType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("ProductID", StringType()) \
    .addColumn("ProductName", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", DoubleType()) \
    .addColumn("Tax", DoubleType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()
    
if RESET_BRONZE:
    spark.sql(f"DELETE FROM {BRONZE_TABLE}")

if RESET_SILVER:
    spark.sql(f"DELETE FROM {SILVER_TABLE}")


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 22, Finished, Available, Finished, False)

**Define raw schemas**

In [21]:
customer_schema = StructType([
    StructField("CustomerID", StringType(), True),
    StructField("FullName", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("City", StringType(), True),
    StructField("MembershipTier", StringType(), True),
    StructField("RecordDate", StringType(), True)
])

sales_schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("OrderLine", IntegerType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Tax", DoubleType(), True)
])


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 23, Finished, Available, Finished, False)

**List files in Incoming**

In [22]:
all_items = notebookutils.fs.ls(INCOMING_DIR)
incoming_csv_files = [x for x in all_items if (not getattr(x, "isDir", False)) and x.name.lower().endswith(".csv")]

if FILES_TO_PROCESS:
    incoming_csv_files = [x for x in incoming_csv_files if x.name in FILES_TO_PROCESS]

print("Files to process:")
for f in incoming_csv_files:
    print("-", f.name)


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 24, Finished, Available, Finished, False)

Files to process:
- Customer_Master_2024_V1.csv
- Sales_Transactions_2024_V1.csv


**Copy each landed file to Bronze archive and load Bronze raw tables**

In [23]:
def bronze_file_already_loaded(bronze_table: str, file_name: str) -> bool:
    return (
        spark.read.table(bronze_table)
        .filter(col("SourceFileName") == file_name)
        .limit(1)
        .count() > 0
    )

customer_bronze_batch_df = None
sales_bronze_batch_df = None

for file_info in incoming_csv_files:
    file_name = file_info.name
    source_path = file_info.path

    year_match = re.search(r"(20\d{2})", file_name)
    year_part = year_match.group(1) if year_match else "unknown"

    archive_folder = f"{BRONZE_ARCHIVE_BASE}/year={year_part}"
    archive_path = f"{archive_folder}/{file_name}"
    notebookutils.fs.mkdirs(archive_folder)

    if file_name.lower().startswith("customer_master"):
        if bronze_file_already_loaded(BRONZE_CUSTOMER_TABLE, file_name):
            print(f"Skipping already loaded Bronze customer file: {file_name}")
            continue

        notebookutils.fs.cp(source_path, archive_path, True)

        raw_df = (
            spark.read.format("csv")
            .option("header", "true")
            .schema(customer_schema)
            .load(source_path)
        )

        one_file_bronze_df = (
            raw_df
            .withColumn("SourceFileName", lit(file_name))
            .withColumn("SourceFilePath", lit(source_path))
            .withColumn("ArchivePath", lit(archive_path))
            .withColumn("IngestedAt", current_timestamp())
            .withColumn("RunId", lit(RUN_ID))
        )

        one_file_bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_CUSTOMER_TABLE)

        if customer_bronze_batch_df is None:
            customer_bronze_batch_df = one_file_bronze_df
        else:
            customer_bronze_batch_df = customer_bronze_batch_df.unionByName(one_file_bronze_df, allowMissingColumns=True)

    elif file_name.lower().startswith("sales_transactions"):
        if bronze_file_already_loaded(BRONZE_SALES_TABLE, file_name):
            print(f"Skipping already loaded Bronze sales file: {file_name}")
            continue

        notebookutils.fs.cp(source_path, archive_path, True)

        raw_df = (
            spark.read.format("csv")
            .option("header", "true")
            .schema(sales_schema)
            .load(source_path)
        )

        one_file_bronze_df = (
            raw_df
            .withColumn("SourceFileName", lit(file_name))
            .withColumn("SourceFilePath", lit(source_path))
            .withColumn("ArchivePath", lit(archive_path))
            .withColumn("IngestedAt", current_timestamp())
            .withColumn("RunId", lit(RUN_ID))
        )

        one_file_bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_SALES_TABLE)

        if sales_bronze_batch_df is None:
            sales_bronze_batch_df = one_file_bronze_df
        else:
            sales_bronze_batch_df = sales_bronze_batch_df.unionByName(one_file_bronze_df, allowMissingColumns=True)
    else:
        print(f"Skipped unsupported file pattern: {file_name}")

print("Customer Bronze rows written this run:", 0 if customer_bronze_batch_df is None else customer_bronze_batch_df.count())
print("Sales Bronze rows written this run:", 0 if sales_bronze_batch_df is None else sales_bronze_batch_df.count())


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 25, Finished, Available, Finished, False)

Customer Bronze rows written this run: 5
Sales Bronze rows written this run: 5


**Build Silver updates from Bronze batches**

In [24]:
if customer_bronze_batch_df is None:
    print("No new customer Bronze files loaded in this run. Customer Silver step skipped.")
else:
    customer_silver_updates_df = (
        customer_bronze_batch_df
        .withColumn("CustomerID", trim(col("CustomerID")))
        .withColumn("FullName", trim(col("FullName")))
        .withColumn("Email", lower(trim(col("Email"))))
        .withColumn("City", trim(col("City")))
        .withColumn("MembershipTier", trim(col("MembershipTier")))
        .withColumn(
            "RecordDate",
            coalesce(
                to_date(trim(col("RecordDate")), "yyyy-MM-dd"),
                to_date(trim(col("RecordDate")), "M/d/yy"),
                to_date(trim(col("RecordDate")), "MM/dd/yy")
            )
        )
        .withColumn("FileName", col("SourceFileName"))
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedTS", current_timestamp())
        .withColumn("ModifiedTS", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .select(
            "CustomerID","FullName","Email","City","MembershipTier","RecordDate",
            "FileName","LoadTimestamp","CreatedTS","ModifiedTS","CreatedBy","ModifiedBy"
        )
        .filter(col("CustomerID").isNotNull() & (col("CustomerID") != ""))
    )

if sales_bronze_batch_df is None:
    print("No new sales Bronze files loaded in this run. Sales Silver step skipped.")
else:
    sales_silver_updates_df = (
        sales_bronze_batch_df
        .withColumn("OrderID", trim(col("OrderID")))
        .withColumn("OrderLine", col("OrderLine").cast("int"))
        .withColumn(
            "OrderDate",
            coalesce(
                to_date(trim(col("OrderDate")), "yyyy-MM-dd"),
                to_date(trim(col("OrderDate")), "M/d/yy"),
                to_date(trim(col("OrderDate")), "MM/dd/yy")
            )
        )
        .withColumn("CustomerID", trim(col("CustomerID")))
        .withColumn("ProductID", trim(col("ProductID")))
        .withColumn("ProductName", trim(col("ProductName")))
        .withColumn("Quantity", col("Quantity").cast("int"))
        .withColumn("UnitPrice", col("UnitPrice").cast("double"))
        .withColumn("Tax", col("Tax").cast("double"))
        .withColumn("FileName", col("SourceFileName"))
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedTS", current_timestamp())
        .withColumn("ModifiedTS", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .select(
            "OrderID","OrderLine","OrderDate","CustomerID","ProductID","ProductName",
            "Quantity","UnitPrice","Tax","FileName","LoadTimestamp",
            "CreatedTS","ModifiedTS","CreatedBy","ModifiedBy"
        )
        .filter(col("OrderID").isNotNull() & col("OrderLine").isNotNull())
    )


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 26, Finished, Available, Finished, False)

**Merge into Silver tables**

In [25]:
if customer_bronze_batch_df is None:
    print("No new customer Bronze batch found. Customer Silver merge skipped.")
else:
    customer_delta = DeltaTable.forName(spark, SILVER_CUSTOMER_TABLE)
    customer_delta.alias("tgt").merge(
        customer_silver_updates_df.alias("src"),
        """
        tgt.CustomerID = src.CustomerID
        AND tgt.RecordDate = src.RecordDate
        """
    ).whenMatchedUpdate(set={
        "FullName": "src.FullName",
        "Email": "src.Email",
        "City": "src.City",
        "MembershipTier": "src.MembershipTier",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "ModifiedTS": "src.ModifiedTS",
        "ModifiedBy": "src.ModifiedBy"
    }).whenNotMatchedInsert(values={
        "CustomerID": "src.CustomerID",
        "FullName": "src.FullName",
        "Email": "src.Email",
        "City": "src.City",
        "MembershipTier": "src.MembershipTier",
        "RecordDate": "src.RecordDate",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "CreatedTS": "src.CreatedTS",
        "ModifiedTS": "src.ModifiedTS",
        "CreatedBy": "src.CreatedBy",
        "ModifiedBy": "src.ModifiedBy"
    }).execute()

if sales_bronze_batch_df is None:
    print("No new sales Bronze batch found. Sales Silver merge skipped.")
else:
    sales_delta = DeltaTable.forName(spark, SILVER_SALES_TABLE)
    sales_delta.alias("tgt").merge(
        sales_silver_updates_df.alias("src"),
        """
        tgt.OrderID = src.OrderID
        AND tgt.OrderLine = src.OrderLine
        """
    ).whenMatchedUpdate(set={
        "OrderDate": "src.OrderDate",
        "CustomerID": "src.CustomerID",
        "ProductID": "src.ProductID",
        "ProductName": "src.ProductName",
        "Quantity": "src.Quantity",
        "UnitPrice": "src.UnitPrice",
        "Tax": "src.Tax",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "ModifiedTS": "src.ModifiedTS",
        "ModifiedBy": "src.ModifiedBy"
    }).whenNotMatchedInsert(values={
        "OrderID": "src.OrderID",
        "OrderLine": "src.OrderLine",
        "OrderDate": "src.OrderDate",
        "CustomerID": "src.CustomerID",
        "ProductID": "src.ProductID",
        "ProductName": "src.ProductName",
        "Quantity": "src.Quantity",
        "UnitPrice": "src.UnitPrice",
        "Tax": "src.Tax",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "CreatedTS": "src.CreatedTS",
        "ModifiedTS": "src.ModifiedTS",
        "CreatedBy": "src.CreatedBy",
        "ModifiedBy": "src.ModifiedBy"
    }).execute()


StatementMeta(, d40a4873-bb3f-43ea-a7b3-41e95c1c1daf, 27, Finished, Available, Finished, False)

**Validate Bronze and Silver output**

In [5]:
print("Customer Bronze count:", spark.read.table(BRONZE_CUSTOMER_TABLE).count())
print("Sales Bronze count:", spark.read.table(BRONZE_SALES_TABLE).count())
print("Customer Silver count:", spark.read.table(SILVER_CUSTOMER_TABLE).count())
print("Sales Silver count:", spark.read.table(SILVER_SALES_TABLE).count())

print("Customer Silver distinct CustomerID + RecordDate count:",
      spark.read.table(SILVER_CUSTOMER_TABLE)
           .select("CustomerID", "RecordDate")
           .distinct()
           .count())

print("Sales Silver distinct OrderID + OrderLine count:",
      spark.read.table(SILVER_SALES_TABLE)
           .select("OrderID", "OrderLine")
           .distinct()
           .count())

display(spark.read.table(SILVER_CUSTOMER_TABLE).orderBy("CustomerID", "RecordDate"))
display(spark.read.table(SILVER_SALES_TABLE).orderBy("OrderDate", "OrderID"))


StatementMeta(, a16e5de9-71f8-4f68-9b09-50316f54cb30, 7, Finished, Available, Finished, False)

Customer Bronze count: 27
Sales Bronze count: 10
Customer Silver count: 16
Sales Silver count: 5
Customer Silver distinct CustomerID + RecordDate count: 16
Sales Silver distinct OrderID + OrderLine count: 5


SynapseWidget(Synapse.DataFrame, 6792acac-f691-4511-9c07-d184161a91ac)

SynapseWidget(Synapse.DataFrame, b3fd8db6-7e87-41c5-8203-dbd26e5f11fe)